# castlegen — end-to-end test of the base level

Runs the base-level sampler of [amdson/Hierwave](https://github.com/amdson/Hierwave) at full size (192 × 192, 802-tile alphabet, 20 checkerboard Gibbs steps, d relaxation, fallback) and checks the validity invariants against an exact BFS.

What this does **not** contain yet: coarse levels, window resampling, the offline pipeline. Without a coarse plan the only source of distance is the gateway, so the castle grows within the light cone of the d relaxation (roughly 30 cells from the gate) and everything outside it is walled by the fallback. Rooms lost is therefore large here by design; the point of this notebook is invariants and throughput, and the plot makes the light cone visible.

Use a GPU runtime (Runtime → Change runtime type → GPU). On CPU a single castle takes about 10 s.

In [ ]:
!git clone -q https://github.com/amdson/Hierwave.git /content/Hierwave 2>/dev/null || (cd /content/Hierwave && git pull -q)
!pip install -q -e /content/Hierwave
import sys; sys.path.insert(0, "/content/Hierwave")
import jax; print(jax.__version__, jax.devices())

## Unit tests

In [ ]:
!cd /content/Hierwave && python -m pytest -q

## One castle at full size

In [ ]:
import time, numpy as np, jax.numpy as jnp
from castlegen import core, base, schedule, e2e

tiles, d, t_first, t_second = e2e.run_one(size=192, castle_id=0, seed=0)
stats = e2e.check(tiles, d)
print(f"compile + run: {t_first:.2f}s   second run: {t_second*1000:.0f} ms")
for k, v in stats.items():
    print(f"  {k:>18}: {v}")

assert stats["violations"] == 0,        "hard terms violated after fallback"
assert stats["half_doors"] == 0,        "dangling door bits after fallback"
assert stats["gateways"] == 1,          "gateway count"
assert stats["unreached_rooms"] == 0,   "a room with d = INF survived the fallback"
assert stats["d_mismatch_vs_bfs"] == 0, "d field differs from exact BFS"
print("all invariants hold")

## Plot: room types and the distance field

In [ ]:
import matplotlib.pyplot as plt
typ, mask, is_room, is_wall, is_gate = [np.asarray(a) for a in core.split_tile(tiles)]
dd = np.asarray(d).astype(float); dd[~is_room] = np.nan
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
img = np.where(is_room, typ, -1).astype(float); img[img < 0] = np.nan
ax[0].imshow(img, cmap="tab20", interpolation="nearest"); ax[0].set_title("room type (walls blank)")
ax[1].imshow(dd, cmap="viridis", interpolation="nearest"); ax[1].set_title("d = BFS distance from the gateway")
gy, gx = np.argwhere(is_gate)[0]
for a in ax: a.plot(gx, gy, "r^", ms=10); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Throughput: a vmapped batch

In [ ]:
batch = 16 if any(dev.platform == "gpu" for dev in jax.devices()) else 2
per_castle = e2e.time_batch(size=192, batch=batch, seed=0)
print(f"batch of {batch}: {per_castle*1000:.1f} ms per castle (base level only, 192 x 192)")

## Light-cone check

The base level alone reaches only cells within the d-propagation horizon. The next step in the plan is the oracle-plan experiment: take the level-1 plan from a reference sample and measure rooms lost; if that is under 1 %, the hierarchy is the remaining work.

In [ ]:
print("max d reached:", stats["max_d"], "  rooms kept:", stats["rooms"])
bands = np.arange(0, stats["max_d"] + 5, 5)
counts, _ = np.histogram(dd[is_room], bins=bands)
for lo, c in zip(bands[:-1], counts):
    print(f"  d in [{lo:3d}, {lo+5:3d}): {c}")